# Tutorial 0: What Is Mechanistic Interpretability?

**Prerequisites:** None. This is the starting point.

**What you'll learn:**
- What mechanistic interpretability is and why it matters
- The key questions mechinterp tries to answer
- How IRTK (the Interpretability Research Toolkit) helps you answer them
- A roadmap for the rest of this tutorial series

---

## The Problem

Large language models (LLMs) like GPT-2, LLaMA, and Claude can write essays, translate languages, and solve math problems. But *how* do they do it? Their internals are billions of floating-point numbers organized into matrices. Understanding what those numbers actually compute is the central challenge of **mechanistic interpretability** (mechinterp).

This matters for several reasons:

1. **Safety**: If we can't understand what a model is doing internally, we can't reliably predict when it will behave dangerously
2. **Science**: Neural networks have discovered novel algorithms for language processing — understanding them advances cognitive science
3. **Engineering**: Understanding failure modes helps us build better models
4. **Trust**: Users and regulators need to know *why* a model produces a given output

## The Approach

Mechanistic interpretability treats neural networks like **reverse-engineering a circuit board**. Instead of treating the model as a black box, we:

1. **Look inside**: Capture every intermediate computation (activations) as the model processes input
2. **Identify components**: Find which parts of the model (attention heads, MLP neurons) are responsible for specific behaviors
3. **Understand algorithms**: Figure out *what computation* those components perform
4. **Verify causally**: Test our understanding by intervening — modifying internal activations and checking if the output changes as predicted

## Key Concepts (Preview)

Here's a preview of the ideas we'll explore in this tutorial series:

| Concept | What it means | Tutorial |
|---------|--------------|----------|
| **Residual stream** | The central "communication bus" that all components read from and write to | T03 |
| **Attention heads** | Components that move information between token positions | T04 |
| **MLP layers** | Components that transform and store knowledge | T05 |
| **Hooks** | Our ability to tap into any intermediate computation | T02 |
| **Activation patching** | Swapping activations between runs to test causal hypotheses | T08 |
| **Circuits** | Small subnetworks responsible for specific behaviors | T09 |

## What is IRTK?

**IRTK** (the **Interpretability Research Toolkit**) is a library built specifically for mechinterp research. Originally inspired by Neel Nanda's TransformerLens, IRTK is a ground-up implementation in JAX using Equinox that has grown far beyond a port:

- **Hook-based activation inspection** — capture and modify any internal activation
- **Pretrained model loading** — load GPT-2, GPT-Neo, Pythia, LLaMA from HuggingFace
- **157+ analysis modules** — from basic logit lens to advanced circuit discovery, SAEs, probing, steering, and more
- **JAX composability** — `jit` for speed, `vmap` for batching, `grad` for differentiation

Let's see it in action.

In [ ]:
# Our first look at a transformer's internals
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig

# Create a small model (we'll use pretrained models later)
cfg = HookedTransformerConfig(
    n_layers=2,     # 2 transformer layers
    d_model=32,     # 32-dimensional residual stream
    n_ctx=64,       # 64 token context window
    d_head=8,       # 8-dimensional attention heads
    n_heads=4,      # 4 attention heads per layer
    d_vocab=100,    # 100 token vocabulary
)
model = HookedTransformer(cfg)
print(f'Created a transformer with {cfg.n_layers} layers, '
      f'{cfg.n_heads} heads, and d_model={cfg.d_model}')

In [ ]:
# Feed it some tokens and capture EVERYTHING that happens inside
tokens = jnp.array([1, 42, 17, 88, 55])  # 5 made-up tokens

logits, cache = model.run_with_cache(tokens)

print(f'Input: {len(tokens)} tokens')
print(f'Output: logits shape = {logits.shape}')  # [seq_len, vocab_size]
print(f'Cached activations: {len(cache)} intermediate values')
print()
print('What we captured (first 10):')
for i, key in enumerate(sorted(cache.keys())):
    shape = cache[key].shape if hasattr(cache[key], 'shape') else '?'
    print(f'  {key}: {shape}')
    if i >= 9:
        print(f'  ... and {len(cache) - 10} more')
        break

That's the core idea: we ran the model and captured every intermediate computation. Each cached activation is a window into what the model is "thinking" at that point in processing.

## A Taste of What's Possible

Let's do three quick demonstrations of what mechinterp can reveal.

In [ ]:
# Demo 1: What does each layer predict?
# The "logit lens" projects intermediate representations through
# the final unembedding matrix to see what each layer would predict.

from irtk.logit_lens import logit_lens

result = logit_lens(model, tokens)
# result['per_layer_top_tokens'] shows the top prediction at each layer
print('Layer-by-layer predictions (token IDs):')
for layer_info in result['per_layer']:
    top = layer_info['top_tokens'][0]  # top prediction at last position
    print(f'  After layer {layer_info["layer"]}: predicts token {top}')

In [ ]:
# Demo 2: Which attention heads matter?
# We can see what each head attends to.

pattern = cache[f'blocks.0.attn.hook_pattern']  # [n_heads, seq, seq]
print(f'Attention patterns shape: {pattern.shape}')
print(f'  = [{cfg.n_heads} heads, {len(tokens)} query positions, {len(tokens)} key positions]')
print()
print('Head 0 attention pattern (which positions does each token attend to?):')
print('  Rows = query position, Columns = key position')
print(np.array(pattern[0]).round(2))

In [ ]:
# Demo 3: How does the residual stream change?
# Track the norm of the residual stream through layers

for l in range(cfg.n_layers):
    resid = cache[f'blocks.{l}.hook_resid_post']
    norm = float(jnp.linalg.norm(resid[-1]))  # last position
    print(f'  After layer {l}: residual stream norm = {norm:.4f}')

print()
print('The residual stream grows as each layer adds its contribution.')
print('Understanding this growth is key to understanding the model.')

## The Tutorial Roadmap

This is the first in a series of tutorials designed to take you from zero to research-ready:

| Tutorial | Topic | What you'll learn |
|----------|-------|------------------|
| **T00** (this one) | What is mechinterp? | Motivation and first look |
| **T01** | Transformers from scratch | Every component explained with code |
| **T02** | Hooks and caching | How to inspect and modify model internals |
| **T03** | The residual stream | The central object of study |
| **T04** | Attention heads decoded | What heads do and how to analyze them |
| **T05** | MLPs and features | How knowledge is stored and retrieved |
| **T06** | Your first investigation | End-to-end guided research project |
| **T07** | The logit lens | Watching predictions form layer by layer |
| **T08** | Activation patching | Testing causal hypotheses about circuits |
| **T09** | Circuit discovery | Finding the minimal circuit for a behavior |
| **T10** | Sparse autoencoders | Decomposing activations into interpretable features |
| **T11** | Advanced techniques | Steering, probing, and beyond |

After completing these tutorials, you'll be equipped to use any of IRTK's 157+ analysis modules and conduct your own mechinterp research.

### Recommended Reading

These tutorials are self-contained, but if you want background reading:

- [A Mathematical Framework for Transformer Circuits](https://transformer-circuits.pub/2021/framework/index.html) (Anthropic, 2021)
- [200 Concrete Problems in Interpretability](https://docs.google.com/spreadsheets/d/1oOdrQ80jDK-aGn-EVdDt3dg65GhmzrvBWzJ6MUZB8n4/) (Neel Nanda)
- [Zoom In: An Introduction to Circuits](https://distill.pub/2020/circuits/zoom-in/) (Olah et al., 2020)

**Next: [T01 — Transformers from Scratch](T01_transformers_from_scratch.ipynb)**